# Notebook 04 — TargetDiff Generation on EGFR Wild-Type (1M17)

**Goals:**
1. Install TargetDiff's PyTorch-Geometric dependencies (CUDA-matched)
2. Inspect the TargetDiff repo to find the sampling script and its argument schema
3. Run conditional generation on the 1M17 pocket — 100 molecules per batch
4. Save each batch to Drive, then merge all batches into `1M17_raw.sdf`
5. Quick validity check

**Runtime: T4 GPU required.** Generation takes ~30–60 minutes per 100 molecules.

**Strategy:** we run generation in **small batches (100 each)** and save after each.
If Colab disconnects, we don't lose everything — just rerun the next batch.

**Output:** `results/generated/1M17_raw.sdf` (target: ≥1000 valid molecules)


In [ ]:
# ── Colab bootstrap + TargetDiff-specific dependencies ─────────────────────
REPO_URL = "https://github.com/Jonathan-Ye-1/egfr-drug-design-eecs6895-final-project.git"

import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = '/content/drive/MyDrive/egfr_diffusion'
if not os.path.exists(os.path.join(PROJECT_ROOT, '.git')):
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_ROOT], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_ROOT, 'pull'], check=False)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.colab_init import setup
PROJECT_ROOT = setup(repo_url=REPO_URL)

# TargetDiff needs torch-geometric + torch-scatter/sparse/cluster — CUDA-matched
import torch
print('PyTorch :', torch.__version__, '| CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Switch Runtime → Change runtime type → T4 GPU'
print('GPU     :', torch.cuda.get_device_name(0))

torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cu' + torch.version.cuda.replace('.', '')[:3]   # e.g. cu121
pyg_wheel = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
print('\nInstalling PyG stack from:', pyg_wheel)

subprocess.run(['pip', 'install', '-q',
                'torch-scatter', 'torch-sparse', 'torch-cluster',
                '-f', pyg_wheel], check=False)
subprocess.run(['pip', 'install', '-q', 'torch-geometric', 'easydict', 'omegaconf'], check=False)

# Register TargetDiff as an importable module
TARGETDIFF = os.path.join(PROJECT_ROOT, 'external', 'targetdiff')
if TARGETDIFF not in sys.path:
    sys.path.insert(0, TARGETDIFF)
print('\nTargetDiff path added to sys.path:', TARGETDIFF)


In [ ]:
# ── Inspect TargetDiff repo structure ──────────────────────────────────────
# Before running generation we verify which scripts / configs are available.
import os

TARGETDIFF = os.path.join(PROJECT_ROOT, 'external', 'targetdiff')
print('=== scripts/ ===')
for f in sorted(os.listdir(os.path.join(TARGETDIFF, 'scripts'))):
    print(' ', f)

print('\n=== configs/ ===')
for f in sorted(os.listdir(os.path.join(TARGETDIFF, 'configs'))):
    print(' ', f)

print('\n=== checkpoints/ ===')
for f in sorted(os.listdir(os.path.join(TARGETDIFF, 'checkpoints'))):
    print(' ', f)


In [ ]:
# ── Look at the sampling script signature + config so we know how to call it ──
import os, subprocess
TARGETDIFF = os.path.join(PROJECT_ROOT, 'external', 'targetdiff')

# Show the sample_for_pocket script header (first 80 lines — usually contains argparse)
script = os.path.join(TARGETDIFF, 'scripts', 'sample_for_pocket.py')
if os.path.exists(script):
    with open(script) as f:
        lines = f.readlines()
    print('=== sample_for_pocket.py (first 80 lines) ===')
    print(''.join(lines[:80]))
else:
    print('sample_for_pocket.py not found; checking alternatives...')
    for f in os.listdir(os.path.join(TARGETDIFF, 'scripts')):
        if 'sample' in f.lower():
            print(' candidate:', f)

# Show the sampling config
cfg = os.path.join(TARGETDIFF, 'configs', 'sampling.yml')
if os.path.exists(cfg):
    print('\n=== configs/sampling.yml ===')
    with open(cfg) as f:
        print(f.read())


In [ ]:
# ── DO NOT RUN YET — we'll fill this in once we've seen the script signature ──
# Based on the output of the previous cell, the exact command depends on
# whether TargetDiff uses argparse or a YAML config file. Typical patterns:
#
#   Pattern A (argparse):
#     python scripts/sample_for_pocket.py \
#         --pdb_path data/pockets/1M17_pocket.pdb \
#         --num_samples 100 \
#         --ckpt_path external/targetdiff/checkpoints/pretrained_diffusion.pt \
#         --result_path results/generated/1M17_batch00
#
#   Pattern B (YAML config):
#     python scripts/sample_diffusion.py configs/sampling.yml \
#         --data_id <pocket_id>
#
# STOP HERE and paste the output of Cell 3 in chat so we can determine the
# exact invocation before burning GPU time.
print("Pause here — paste the output of the previous cell to ChatGPT/Claude "
      "so we can construct the right sampling command for your TargetDiff version.")
